<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# Company PAC Superset Construction Project Overview

## Task:
Integrate FEC political committee data with Snowflake contribution data to classify companies based on PAC activity and donation behavior.

---

## Dataset:
- FEC Committee Master (.txt files across election cycles)
- Snowflake PAC contribution dataset
- Company universe dataset
- Company–ticker crosswalk dataset

---

## What This Pipeline Does?

**Load:**  
Reads raw FEC committee master files and Snowflake PAC activity data.

**Filter:**  
Identifies corporate PACs by:
- committee type
- naming patterns (e.g., INC, CORP, PAC)

**Normalize:**  
Cleans committee names and company names by:
- lowercasing
- removing punctuation
- standardizing suffixes

**Merge:**  
Combines:
- FEC PAC data
- Snowflake contribution data

**Classify:**  
Determines PAC status for each company:
- has PAC
- has donations
- no donations

**Aggregate:**  
Rolls up PAC activity to the company level.

---

## Libraries Used

**Packages:**
- pandas  
- numpy  
- re  

---

## Pipeline

### Step 1: FEC Data Loading
- Load committee master files (multiple election cycles)  
- Assign column names  
- Combine into a unified dataset  

---

### Step 2: Corporate PAC Filtering
- Filter committees relevant to corporations  
- Remove irrelevant political committees  

---

### Step 3: Snowflake Data Integration
- Load Snowflake PAC activity dataset  
- Identify PACs with contribution records  

---

### Step 4: Name Normalization
- Clean and standardize committee names  
- Align naming across FEC and Snowflake datasets  

---

### Step 5: Matching & Merging
- Merge FEC PACs with Snowflake contribution data  
- Identify matched and unmatched PACs  

---

### Step 6: Donation Flagging
- Create flags:
  - `has_pac`
  - `has_pac_donations`  

---

### Step 7: Company-Level Aggregation
- Group PAC data by company  
- Compute final PAC status  

---

### Step 8: Final Classification
Assign each company to one of:

- `NO_PAC`  
- `PAC_DONATED`  
- `PAC_NO_DONATIONS`  

---

## Outputs

### CSV Files:
- `fec_committee_master.csv` → unified FEC dataset  
- `fec_pac_master.csv` → filtered corporate PACs  
- `fec_pacs_with_donation_flag.csv` → PACs with donation info  
- `fec_pacs_no_donations.csv` → PACs without donations  
- `company_pac_donation_status.csv` → company-level PAC summary  
- `company_pac_superset_U.csv` → final integrated dataset  

---

## Goal

Create a unified dataset that links **companies to their political activity**, enabling:

- PAC activity classification  
- corporate political behavior analysis  
- integration with executive and company datasets  

---

# Imports

In [1]:
import pandas as pd
import re

# Import the FEC Data

In [2]:
df = pd.read_csv("fec_pacs_with_donation_flag.csv", dtype=str)
df["TOTAL_AMOUNT"] = pd.to_numeric(df["TOTAL_AMOUNT"], errors="coerce").fillna(0)
df["HAS_DONATED"] = df["TOTAL_AMOUNT"] > 0

# Group by controlling organization

In [3]:
company_rollup = (
    df
    .groupby("CONNECTED_ORG_NM", dropna=True)
    .agg(
        pac_count=("CMTE_ID", "nunique"),
        active_pac_count=("HAS_DONATED", "sum"),
        has_any_pac_donated=("HAS_DONATED", "any"),
        total_pac_amount=("TOTAL_AMOUNT", "sum")
    )
    .reset_index()
)

#Add lebels
company_rollup["has_pac"] = True
company_rollup["has_pac_donations"] = company_rollup["has_any_pac_donated"]

In [4]:
company_rollup.head(10)

,CONNECTED_ORG_NM,pac_count,active_pac_count,has_any_pac_donated,total_pac_amount,has_pac,has_pac_donations
0,1ST SOURCE CORPORATION,1,0,False,0.00,True,False
1,"21ST CENTURY ONCOLOGY, INC",1,0,False,0.00,True,False
2,3M COMPANY,1,1,True,63250.00,True,True
3,"A DUDA & SONS, INC POLITICAL ACTION COMMITTEE",1,0,False,0.00,True,False
4,A. O. SMITH CORPORATION,1,1,True,3000.00,True,True
5,"AARON'S, INC.",1,0,False,0.00,True,False
6,ABBVIE,1,1,True,22549.87,True,True
7,"ABIOMED, INC.",1,0,False,0.00,True,False
8,ACADIAN AMBULANCE SERVICE EMPLOYEE FEDERAL PO...,1,0,False,0.00,True,False
9,ACXIOM LLC,1,0,False,0.00,True,False


# Create a clean P-set file

In [5]:
def looks_like_pac(s: str) -> bool:
    s = "" if s is None else str(s).lower()
    return bool(re.search(r"\b(pac|political action committee)\b", s))

company_rollup_clean = company_rollup[
    ~company_rollup["CONNECTED_ORG_NM"].map(looks_like_pac)
].copy()

In [6]:
removed = len(company_rollup) - len(company_rollup_clean)

print("Company rollup (raw):", company_rollup.shape)
print("Company rollup (clean):", company_rollup_clean.shape)
print("Removed (orgs that looked like PAC names):", removed)

print("Companies with PAC(s) but NO donations:",
      (company_rollup_clean["has_pac_donations"] == False).sum())

print("Companies with at least one donating PAC:",
      (company_rollup_clean["has_pac_donations"] == True).sum())

Company rollup (raw): (796, 7)
Company rollup (clean): (720, 7)
Removed (orgs that looked like PAC names): 76
Companies with PAC(s) but NO donations: 469
Companies with at least one donating PAC: 251


# Outputs

In [7]:
company_rollup_clean.to_csv("company_pac_donation_status.csv", index=False)

# Optional: preview
company_rollup_clean.head(10)

,CONNECTED_ORG_NM,pac_count,active_pac_count,has_any_pac_donated,total_pac_amount,has_pac,has_pac_donations
0,1ST SOURCE CORPORATION,1,0,False,0.00,True,False
1,"21ST CENTURY ONCOLOGY, INC",1,0,False,0.00,True,False
2,3M COMPANY,1,1,True,63250.00,True,True
4,A. O. SMITH CORPORATION,1,1,True,3000.00,True,True
5,"AARON'S, INC.",1,0,False,0.00,True,False
6,ABBVIE,1,1,True,22549.87,True,True
7,"ABIOMED, INC.",1,0,False,0.00,True,False
9,ACXIOM LLC,1,0,False,0.00,True,False
10,AECOM,1,1,True,6700.00,True,True
11,AEROJET ROCKETDYNE INC.,1,0,False,0.00,True,False


# Brand Extraction

In [8]:
sf = pd.read_csv("snowflake_pac_activity.csv", dtype=str)

# Normalization

In [9]:
def norm_committee_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)  # remove punctuation
    s = re.sub(r"\b(pac|political|action|committee|fund|the|and|of|for|employees)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

fec_p["committee_name_clean"] = fec_p["CMTE_NM"].map(norm_committee_name)
sf["committee_name_clean"] = sf["COMMITTEE_NAME"].map(norm_committee_name)

NameError: name 'fec_p' is not defined

# Join and flag dontations

In [ ]:
sf["TOTAL_AMOUNT"] = pd.to_numeric(sf["TOTAL_AMOUNT"], errors="coerce").fillna(0)

# Left join FEC to Snowflake Data

In [ ]:
merged = fec_p.merge(
    sf[["committee_name_clean", "TOTAL_AMOUNT", "HAS_DONATED", "FIRST_CYCLE", "LAST_CYCLE"]],
    on="committee_name_clean",
    how="left",
    suffixes=("_fec", "_sf")
)

merged["TOTAL_AMOUNT"] = merged["TOTAL_AMOUNT"].fillna(0)
merged["HAS_DONATED"] = merged["HAS_DONATED"].fillna(False)

# Output

In [ ]:
merged.to_csv("fec_pacs_with_donation_flag.csv", index=False)
no_donations = merged[merged["TOTAL_AMOUNT"] == 0]
no_donations.to_csv("fec_pacs_no_donations.csv", index=False)

In [ ]:
print("Total FEC corporate PACs:", len(merged))
print("Matched to Snowflake PACs:", (merged["TOTAL_AMOUNT"] > 0).sum())
print("No donation activity:", (merged["TOTAL_AMOUNT"] == 0).sum())